# 1IHM - Norwalk virus (norovirus) capsid, as a T=3 model

The deposited entry is the T=3 icosahedral capsid: 180 copies of the VP1 coat
protein. The target assembly is a 180-mer, and the question is whether a NERDSS
model built straight from the structure grows a *shell* rather than a blob.

Run without ProAffinity: `predict_affinity` stays at its default `False`, so
every binding reaction gets the same generic rate constants.

## Why this example uses `chain_grouping_matching_mode="sequence_structure"`

All 180 chains are one sequence, so every other grouping path collapses them
into a single molecule type. That single type is not a faithful subunit:

* **It has the wrong valence.** ioNERDSS finds 13 interfaces across the
  assembly, but no real chain uses more than 5 of them. The 180 chains fall
  into three classes of exactly 60, each using a near-disjoint set:

  | class | interfaces used |
  |---|---|
  | A | `AA1b AA1f AA2 AA3f AA4f` |
  | B | `AA2 AA3b AA5f AA6f AA7f` |
  | C | `AA2 AA4b AA5b AA6b AA7b` |

  That is T=3 quasi-equivalence: 60 A + 60 B + 60 C, sharing only `AA2`, the
  2-fold contact. Collapsed into one type, every simulated subunit can act as
  A *and* B *and* C at once and make contacts no real VP1 makes.

* **Its geometry is spliced.** The representative chain carries only 5 of the
  13 sites, so the other 8 are taken from chains in the other two conformations
  and the build logs `Falling back to mixed-frame instances (RISKY)`. The
  radii and the within-conformer angles are still exact, but the relative
  geometry of any cross-conformer pair of sites was never measured and is
  therefore arbitrary.

The three classes really are different shapes, so a rigid template cannot hold
all three at once. Superimposing the conformers gives Ca RMSDs of A-B 0.99 A,
A-C 2.95 A, B-C 3.02 A. Each domain is internally rigid (P-domain RMSD
0.22-0.57 A); the difference is the S-P hinge, which swings the P domain by a
mean of 10.4 A between A and C. Sequence cannot see that, and structure alone
cannot see that they are the same protein -- so grouping has to require both.

`rmsd_threshold` picks how finely the conformers are split:

| threshold | groups |
|---|---|
| 3.5 A | 1 x 180 - everything merged again |
| 2.0 A | 2 - `{A,B}` and `{C}`, since A and B differ by only 0.99 A |
| **0.5 A** | **3 x 60 - one type per quasi-equivalent conformer** |

0.5 A is the value this notebook uses. It gives three `.mol` files with
**5 interfaces each**, zero mixed-frame warnings, and 8 reactions including the
flat `C(cc1) + C(cc1)` homodimer alongside the A-B contacts -- the classic
A/B-and-C/C dimer pair of a T=3 capsid. It is also about 17x cheaper to
simulate than the collapsed model, because reaction matching scales with the
number of interfaces per molecule.

In [ ]:
# Path handling (standard library)
from pathlib import Path

# Core imports
import ionerdss as ion
from ionerdss import build_system_from_pdb

# For visualizations
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
pdb_id = "1ihm"

# Build the system using simplified API
# This takes ~60 s: the biological assembly is a 77 MB mmCIF with 180 chains,
# and sequence_structure superimposes chains pairwise on top of that.
system = build_system_from_pdb(
    source=pdb_id,
    workspace_path=f"{pdb_id}_dir",

    # The asymmetric unit is only the A/B/C protomer and yields 3 binding
    # reactions -- not enough contacts to close a shell. The biological
    # assembly carries all 180 chains and recovers the full contact set.
    pdb_file_format="bioassembly1",

    # 0.6 nm / 3 residues is well conditioned here: 0.4 nm finds a single
    # contact, 0.8 nm over-splits into 17 interfaces / 11 reactions, while
    # anything in between gives a stable 13 interfaces / 7 reactions on the
    # collapsed model whether the residue cutoff is 2 or 3.
    interface_detect_distance_cutoff=0.6,
    interface_detect_n_residue_cutoff=3,

    # One sequence, three conformations -> require both tests. See above:
    # 0.5 A separates all three quasi-equivalent conformers, 2.0 A would
    # merge A with B, 3.5 A would merge all three back into one type.
    chain_grouping_matching_mode="sequence_structure",
    chain_grouping_seq_threshold=0.5,
    chain_grouping_rmsd_threshold=0.5,

    # Capsid: project the subunits onto a best-fit sphere so the association
    # geometry is consistent with a closed shell.
    is_on_sphere=True,

    # Two capsids' worth at 1:1:1, which is what T=3 requires.
    # 360 subunits in a 500 nm box is ~4.8 uM.
    molecule_counts={"A": 120, "B": 120, "C": 120},
    nerdss_water_box=[500.0, 500.0, 500.0],
    nerdss_n_itr=300000,

    # NERDSS indexes its transition matrix by how many copies of one molecule
    # type sit in a complex, so this must be >= that count; leaving it None
    # picks the molecule count automatically.
    count_transition=True,
    transition_matrix_size=None,
)

print(system.get_summary())

In [ ]:
# One .mol per quasi-equivalent conformer, each carrying only its own 5 sites
workspace_path = Path(f"{pdb_id}_dir")
nerdss_dir = workspace_path / "nerdss_files"

print("Molecule templates:")
for f in sorted(nerdss_dir.glob("*.mol")):
    sites = [ln.split()[0] for ln in f.read_text().splitlines()
             if ln[:2].islower() and len(ln.split()) == 4 and ln.split()[0] != "COM"]
    print(f"  {f.name:<10} {len(sites)} interfaces: {' '.join(sites)}")

# The T=3 contact set: A-A, A-B x2, A-C, B-C x3, and the flat C-C homodimer
print("\nReaction network:")
print((nerdss_dir / "parms.inp").read_text().split("start reactions")[1])

In [ ]:
# run NERDSS with subprocess
import subprocess

# Check if NERDSS is available
# nerdss_cmd should be replaced with the actual path to the NERDSS executable
nerdss_cmd = "PATH_TO_NERDSS_REPO/bin/nerdss"
nerdss_path = Path(nerdss_cmd).expanduser() # replaces tilde with appropriate user home path

if nerdss_path.exists():

    # ~160 s for this model. The single-type model takes ~2700 s for the same
    # 300k iterations, because each subunit there carries 13 interfaces.
    result = subprocess.run(
        f"{nerdss_cmd} -f parms.inp",
        shell=True,
        cwd=f"{pdb_id}_dir/nerdss_files",
        capture_output=True,
        text=True
    )

    if result.returncode == 0:
        print("NERDSS simulation completed!")
        print(f"\nCheck {pdb_id}_dir/nerdss_files/ for output files")
    else:
        print(f"NERDSS simulation failed (returncode {result.returncode})")
        print(result.stderr[:500])
else:
    print("NERDSS not found at:", nerdss_cmd)

## Reading the result

With three molecule types the size histogram records the A/B/C composition of
every complex, so the growth curve can be checked against what a T=3 shell
requires: subunits taken up in a 1:1:1 ratio, all the way to 60:60:60.

The imbalance below is `sum |f_i - 1/3|` over the three fractions in the
largest complex, so 0 is a perfect 1:1:1 and larger means the complex is
enriched in one conformer.

In [ ]:
import re
from collections import Counter

hist = Path(f"{pdb_id}_dir/nerdss_files/DATA/histogram_complexes_time.dat").read_text()

times, largest, imbalance, composition = [], [], [], []
for block in hist.split("Time (s): ")[1:]:
    lines = block.strip().splitlines()
    comps = []
    for ln in lines[1:]:
        m = re.match(r"\s*(\d+)\s+(.*)", ln)
        if not m:
            continue
        comp = {k: int(v) for k, v in re.findall(r"([ABC]):\s*(\d+)\.", m.group(2))}
        if comp:
            comps.append(comp)
    if not comps:
        continue
    big = max(comps, key=lambda c: sum(c.values()))
    total = sum(big.values())
    times.append(float(lines[0]))
    largest.append(total)
    composition.append([big.get(k, 0) for k in "ABC"])
    frac = np.array(composition[-1]) / total
    imbalance.append(np.abs(frac - 1 / 3).sum())

times, largest = np.array(times), np.array(largest)
imbalance, composition = np.array(imbalance), np.array(composition)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(times, largest, label="largest complex")
for i, k in enumerate("ABC"):
    axes[0].plot(times, composition[:, i], lw=1, ls="--", label=f"{k} in it")
axes[0].axhline(180, c="k", ls=":", lw=1, label="T=3 capsid (180)")
axes[0].set_xlabel("time (s)"); axes[0].set_ylabel("subunits")
axes[0].set_title("1IHM: growth of the largest complex")
axes[0].legend(fontsize=8)

grew = largest >= 10
axes[1].plot(largest[grew], imbalance[grew], lw=1)
axes[1].set_xlabel("complex size (subunits)")
axes[1].set_ylabel("A:B:C imbalance  (0 = perfect 1:1:1)")
axes[1].set_title("is it growing as a shell?")

plt.tight_layout()
plt.show()

for lo, hi in [(10, 50), (50, 100), (100, 180)]:
    sel = (largest >= lo) & (largest < hi)
    if sel.any():
        print(f"size {lo:3d}-{hi:3d}: mean imbalance {imbalance[sel].mean():.3f}")
a, b, c = composition[-1]
print(f"\nlargest complex: {largest[-1]} subunits  A={a} B={b} C={c} "
      f"({largest[-1] / 180:.0%} of a T=3 capsid)")

## What to expect

**Shell growth is captured.** The A:B:C ratio of the largest complex converges
on 1:1:1 as it grows -- mean imbalance falls from ~0.16 at 10-50 subunits to
~0.04 above 100 -- and the run closes ~290 rings (`Nloops` in
`DATA/bound_pair_time.dat`). The growing edge takes up conformers in the ratio
the lattice demands rather than accreting whatever it meets, which is the
thing the collapsed single-type model cannot represent at all.

**Completion is not.** The run ends with 4-6 partial shells of 70-125 subunits
and no free monomers, the largest reaching ~55-70% of a capsid. Weakening every
bond 100x (`parms_overrides={"force_off_ratekb": 813.0}`) tightens the
stoichiometry to ~0.01 imbalance and pushes the largest to ~124, and an 8x
dilution over 16x more simulated time still ends at 4 complexes -- so the
shortfall is not a lack of time.

The reason is that without ProAffinity every interface is assigned the same
default binding energy of -16 RT (`Kd` ~ 112 nM). A dimer is then as stable
per bond as the finished lattice, so there is no nucleation barrier: nuclei
appear everywhere at once and fragment the monomer pool before any one shell
can finish. Real capsid assembly relies on the opposite -- individually weak
contacts, with avidity only past a critical nucleus. No single global off-rate
or concentration can create that hierarchy, because it requires *different*
interfaces to have *different* affinities, which is exactly what ProAffinity
supplies and what this example leaves out.